# Config

> Where the tunnel goes, what we assume about it, and where files live.


In [ ]:
#| default_exp config

In [ ]:
#| export
import os
from pathlib import Path

import matplotlib as mpl
import pyproj
from dotenv import load_dotenv
from shapely.geometry import LineString, Point

In [ ]:
from fastcore.test import test_close, test_eq

## Files

Downloads are cached in `data/raw/`, intermediate results go in `data/derived/`, and the blog's charts in `figures/`.


In [ ]:
#| export
ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
RAW, DERIVED, FIGURES = ROOT / "data" / "raw", ROOT / "data" / "derived", ROOT / "figures"
for _d in (RAW, DERIVED, FIGURES):
    _d.mkdir(parents=True, exist_ok=True)

Two inputs need an HM Land Registry account, so we can't download them for you. Say where they are in a `.env` file
at the project root (copy `.env.example`):

- `NPS_DIR`: the National Polygon Service shapefiles (`LR_POLY_FULL_*.shp`), i.e. title plans.
- `COD_DIR`: the CCOD and OCOD files of company owners (`CCOD_FULL_*.zip`, `OCOD_FULL_*.zip`). Defaults to `data/`.


In [ ]:
#| export
load_dotenv(ROOT / ".env")
NPS_DIR = Path(os.environ.get("NPS_DIR", ROOT / "data" / "nps"))
COD_DIR = Path(os.environ.get("COD_DIR", ROOT / "data"))

In [ ]:
test_eq((ROOT / "pyproject.toml").exists(), True)

## Coordinates

Points are given as (longitude, latitude), which is how maps label them. Distances and areas are measured on the
British National Grid (BNG), whose units are metres.


In [ ]:
#| export
BNG, WGS84 = "EPSG:27700", "EPSG:4326"
_to_bng = pyproj.Transformer.from_crs(WGS84, BNG, always_xy=True)


def to_bng(lonlat: tuple[float, float]) -> tuple[float, float]:
    "Convert (lon, lat) to BNG (x, y) metres."
    return _to_bng.transform(*lonlat)

Big Ben is at grid reference TQ 30268 79640:


In [ ]:
test_close(to_bng((-0.12463, 51.50073)), (530_268, 179_640), eps=10)

## The route

The tunnel runs from Durand's Wharf park on Rotherhithe Street to the West India Docks impound lock on Westferry
Road. One Canada Square stands in for "the jobs".


In [ ]:
#| export
SOUTH_PORTAL = (-0.032771, 51.501797)  # Durand's Wharf park, Rotherhithe
NORTH_PORTAL = (-0.026450, 51.503040)  # impound lock land, Westferry Road
CANARY_WHARF = (-0.019570, 51.504970)  # One Canada Square


def tunnel_line() -> LineString:
    "Straight line between the portals, in BNG."
    return LineString([to_bng(SOUTH_PORTAL), to_bng(NORTH_PORTAL)])

A bored tunnel can't quite follow a straight line, so we allow 10% extra.


In [ ]:
#| export
TUNNEL_DETOUR = 1.10  # bored length / straight-line distance


def tunnel_length_m() -> float:
    "Assumed length of the tunnel as built."
    return tunnel_line().length * TUNNEL_DETOUR

In [ ]:
test_close(tunnel_line().length, 460, eps=5)
test_close(tunnel_length_m(), 506, eps=5)

## Travel assumptions


In [ ]:
#| export
WALK_KMH = 4.8
CYCLE_KMH = 15.0
PORTAL_DELAY_MIN = 1.5  # at each end: waiting for and riding the lift

## Study area

Street networks and house sales cover Rotherhithe, Bermondsey, Deptford, the Isle of Dogs and Wapping. The
land readjustment area is a circle around the south portal.


In [ ]:
#| export
STUDY_BBOX = (-0.085, 51.470, 0.012, 51.522)  # (west, south, east, north)
CATCHMENT_M = 750


def catchment() -> Point:
    "The readjustment circle around the south portal, in BNG. Notebook 02 trims it to the Rotherhithe bank."
    return Point(to_bng(SOUTH_PORTAL)).buffer(CATCHMENT_M)

In [ ]:
test_close(catchment().area / 1e4, 3.1416 * 0.75**2 * 100, eps=1)  # ~177 ha

## Money

All money is in 2025 prices.


In [ ]:
#| export
PRICE_YEAR = 2025

## Charts

One palette and style for every figure in the blog.


In [ ]:
#| export
COLOURS = {
    "tunnel": "#e8673a", "southwark": "#2b78d6", "housing": "#ee6a37", "other": "#1db07f", "rest": "#b8b8b0", "private": "#dcdcd6",
    "water": "#d6e4ef", "land": "#f0efea", "park": "#cfe3c8", "jubilee": "#7b868c", "text": "#555555",
}  # fmt: skip
BACKGROUND = "#fbfbf9"


def use_style():
    "Apply the blog's chart style to matplotlib."
    mpl.rcParams.update({
        "figure.facecolor": BACKGROUND, "axes.facecolor": BACKGROUND, "savefig.facecolor": BACKGROUND,
        "savefig.dpi": 150, "savefig.bbox": "tight", "font.size": 11, "legend.frameon": False,
        "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#aaaaaa",
        "axes.labelcolor": COLOURS["text"], "xtick.color": COLOURS["text"], "ytick.color": COLOURS["text"],
    })  # fmt: skip